In [2]:
import torch
import datasets
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
import utils

/u/pmirtaheri/.conda/envs/simons/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Model

In [3]:
cache_dir = "/work/hdd/bbjr/huggingface/"

def get_model(model_name, device="auto"):
    """Loads a model and tokenizer from Hugging Face."""
    print(f"Loading model: {model_name}")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir, trust_remote_code=True, use_fast=True)
    
    # Add padding token if it doesn't exist
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Determine the best dtype and device configuration
    if torch.cuda.is_available():
        dtype = torch.bfloat16
        device_map = "auto"
    else:
        dtype = torch.float32
        device_map = "cpu"
        print("CUDA not available, using CPU")
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True,
        cache_dir=cache_dir
    )
    
    # Set model to evaluation mode
    model.eval()
    
    print(f"Model {model_name} loaded successfully.")
    print(f"Model device: {next(model.parameters()).device}")
    print(f"Model dtype: {next(model.parameters()).dtype}")
    
    return model, tokenizer

In [4]:
model_name = 'Qwen/Qwen2.5-7B-Instruct'
model, tokenizer = get_model(model_name)

Loading model: Qwen/Qwen2.5-7B-Instruct


Loading checkpoint shards: 100%|██████████| 4/4 [00:32<00:00,  8.15s/it]

Model Qwen/Qwen2.5-7B-Instruct loaded successfully.
Model device: cuda:0
Model dtype: torch.bfloat16


## Load Dataset

In [66]:
data = datasets.load_dataset("cais/mmlu", "all")
data = data['test']

In [26]:
# import numpy as np
# print(np.unique(data['test']['subject']))
# test_data = data['test']
# data = data[data['subject'] == 'college_mathematics']
# print(test_data)
# college_math = test_data.filter(lambda x: x['subject'] == 'college_mathematics')

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 14042
})


Filter: 100%|██████████| 14042/14042 [00:00<00:00, 105368.04 examples/s]


## Forward Pass

In [74]:
def generate_answer(base_prompt, use_cot):
    if use_cot:
        prompt = base_prompt + "Think step by step. Then, based on your reasoning, provide the single most likely answer choice in the format Final answer: X, where X is A, B, C, or D."
        max_new_tokens = 2048
    else:
        prompt = base_prompt + "Answer with a single letter (A, B, C, or D):"
        max_new_tokens = 1
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        generation_kwargs = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "max_new_tokens": max_new_tokens,  
            "do_sample": True,
            "temperature": 0.6,
            "pad_token_id": tokenizer.eos_token_id,
        }
        outputs = model.generate(**generation_kwargs)
        generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return generated_text

In [75]:
for example in tqdm(test_data):
    question = example['question']
    choices = example['choices']
    correct_answer_idx = example['answer']
    base_prompt = f"Question: {question}\n"
    for i, choice in enumerate(choices):
        base_prompt += f"{chr(65+i)}. {choice}\n"
    print('CoT Answer:', generate_answer(base_prompt, True))
    print('No CoT Answer:', generate_answer(base_prompt, False))
    print('Correct Answer:', chr(ord('A') + correct_answer_idx))

  0%|          | 1/14042 [00:18<70:18:55, 18.03s/it]

CoT Answer:  To find the degree of the field extension \( \mathbb{Q}(\sqrt{2}, \sqrt{3}, \sqrt{18}) \) over \( \mathbb{Q} \), we need to understand the structure of this extension and how it builds up.

1. **Identify the elements**: The field extension includes \( \sqrt{2} \), \( \sqrt{3} \), and \( \sqrt{18} \). Note that \( \sqrt{18} = \sqrt{9 \cdot 2} = 3\sqrt{2} \).

2. **Simplify the extension**: Since \( \sqrt{18} = 3\sqrt{2} \), adding \( \sqrt{18} \) to the field \( \mathbb{Q}(\sqrt{2}, \sqrt{3}) \) does not introduce a new element because \( 3\sqrt{2} \) is already in \( \mathbb{Q}(\sqrt{2}) \). Therefore, \( \mathbb{Q}(\sqrt{2}, \sqrt{3}, \sqrt{18}) = \mathbb{Q}(\sqrt{2}, \sqrt{3}) \).

3. **Degree of \( \mathbb{Q}(\sqrt{2}, \sqrt{3}) \) over \( \mathbb{Q} \)**:
   - First, consider the extension \( \mathbb{Q}(\sqrt{2}) \) over \( \mathbb{Q} \). The minimal polynomial of \( \sqrt{2} \) over \( \mathbb{Q} \) is \( x^2 - 2 \), which is irreducible over \( \mathbb{Q} \). Thus, t

  0%|          | 2/14042 [00:30<57:20:27, 14.70s/it]

CoT Answer:  To find the index of the subgroup generated by \( p \) in \( S_5 \), we need to follow these steps:

1. **Determine the order of \( p \)**:
   - The permutation \( p \) is given as \( p = (1, 2, 5, 4)(2, 3) \).
   - First, simplify the permutation:
     - \( (1, 2, 5, 4) \) means \( 1 \to 2 \to 5 \to 4 \to 1 \).
     - \( (2, 3) \) means \( 2 \to 3 \to 2 \).
   - Combining these, we get:
     - \( 1 \to 2 \to 3 \to 5 \to 4 \to 1 \).

   This can be written as a single cycle:
   - \( p = (1, 2, 3, 5, 4) \).

2. **Find the order of the cycle**:
   - The length of the cycle \( (1, 2, 3, 5, 4) \) is 5.

3. **Order of the subgroup generated by \( p \)**:
   - The order of the subgroup generated by \( p \), denoted as \( \langle p \rangle \), is equal to the order of \( p \), which is 5.

4. **Calculate the index of \( \langle p \rangle \) in \( S_5 \)**:
   - The index of a subgroup \( H \) in a group \( G \) is given by \( [G : H] = \frac{|G|}{|H|} \).
   - Here, \( |S_5| = 5!

  0%|          | 3/14042 [01:23<125:57:30, 32.30s/it]

...

CoT Answer:  Step 1: Analyzing Statement 1
- A homomorphism is a function between two algebraic structures (like groups) that preserves the structure's operations.
- If a homomorphism is one-to-one (injective), it means each element in the domain maps to a unique element in the codomain, and no two different elements in the domain map to the same element in the codomain.
- For a homomorphism to be one-to-one, its kernel must consist only of the identity element. This is because if there were any other element in the kernel, say \(a \neq e\), then \(f(a) = e\) in the codomain, which would violate the one-to-one property since \(f(e) = e\) as well.

Step 2: Analyzing Statement 2
- The image of a group under a homomorphism is a subgroup of the codomain.
- According to Lagrange's theorem, the order (number of elements) of any subgroup divides the order of the group.
- A group of 6 elements can have subgroups of orders that are divisors of 6, which are 1, 2, 3, and 6.
- Therefore, the image

  1%|          | 87/14042 [19:58<44:30:37, 11.48s/it]

CoT Answer:  To determine the truth of each statement, let's analyze them one by one.

### Statement 1:
"If \( H \) and \( K \) are subgroups of a group \( G \), then \( |HK| = \frac{|H||K|}{|H \cap K|} \)."

This statement is true. The formula given is a well-known result in group theory known as the product formula for the order of the product of two subgroups. Here's why:

- \( HK \) is the set of all products \( hk \) where \( h \in H \) and \( k \in K \).
- Each element in \( HK \) can be written uniquely as \( h(kh^{-1})k' \) where \( h \in H \), \( k \in K \), and \( k' \in K \cap H \).
- The number of such unique products is given by \( \frac{|H||K|}{|H \cap K|} \).

Thus, Statement 1 is true.

### Statement 2:
"A group of order \( 2p \) where \( p \) is an odd prime is cyclic or isomorphic to \( D_p \)."

This statement is also true. According to Sylow's Theorems and some group theory results, a group of order \( 2p \) (where \( p \) is an odd prime) has either a normal Sylow 

  1%|          | 87/14042 [20:24<54:34:29, 14.08s/it]


KeyboardInterrupt: 